In [22]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

True

###  connection setup  


In [7]:
schema = "sample_library"
user = "root"
password = os.getenv("MYSQL_PASSWORD")
host = "localhost"
port = 3306

connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{schema}"
engine = create_engine(connection_string)

In [8]:
from sqlalchemy import create_engine

engine = create_engine(connection_string)

In [70]:
#attempt to check if a book with a specific ISBN already exists after seeding the DB
isbn_to_check = "9780451524935"
existing_book = pd.read_sql(
    f"SELECT * FROM books WHERE isbn = '{isbn_to_check}'",
    con=engine
)
if existing_book.empty:
    print(f"No book found with ISBN {isbn_to_check}.")
else:
    print("Book found:")
    display(existing_book)

Book found:


,isbn,title,publisher,published_year
0,9780451524935,1984,Penguin Books,1949


### Read Operations

In [81]:
def read_friends():
    query = "SELECT * FROM friends"
    return pd.read_sql(query, con=engine)

In [139]:
def read_books(available_only=False):
    if available_only:
        query = """
            SELECT b.*
            FROM books AS b
            LEFT JOIN loans AS l
                ON b.isbn = l.isbn
                AND l.status = 'out'
            WHERE l.loan_id IS NULL
        """
    else:
        query = "SELECT * FROM books"

    return pd.read_sql(query, con=engine)

In [141]:
read_books(available_only=True)

,isbn,title,publisher,published_year
0,9780062073488,And Then There Were None,William Morrow,1939
1,9780062316097,Sapiens,Harper,2011
2,9780451524935,1984,Penguin Books,1949
3,9781400033416,Beloved,Vintage,1987


In [79]:
def read_loans():
    return pd.read_sql("loans", con=engine)

### Create Operations

In [104]:
def create_friend(name, email=None, phone=None, notes=None):
    query = """
        INSERT INTO friends (full_name, email, phone, notes) 
        VALUES (:name, :email, :phone, :notes)
    """

    with engine.begin() as connection:
        connection.execute(text(query), {
            "name": name, 
            "email": email, 
            "phone": phone, 
            "notes": notes
        })

    print(f"Human name : {name} successfully added!")

In [103]:
create_friend("Semira", "semira@example.com", "123-456-7890", "Loves debugging code")

Human Semira successfully added!


In [105]:
read_friends()

,friend_id,full_name,email,phone,notes
0,1,Liana Cruz,liana@email.com,555-0101,NaN
1,2,Maya Chen,maya.chen@email.com,555-0102,NaN
2,3,Diego Alvarez,diego.alvarez@email.com,555-0103,NaN
3,4,Priya Nair,priya.nair@email.com,555-0104,NaN
4,5,Sam O'Brien,sam.obrien@email.com,555-0105,NaN
5,6,Yusuf Demir,yusuf.demir@email.com,555-0106,NaN
6,7,Ana Kowalski,ana.kowalski@email.com,555-0107,NaN
7,11,Semira,semira@example.com,123-456-7890,Loves debugging code


In [145]:
# Suggested version for the current sample_library schema.
def create_loan_by_name(
    isbn,
    friend_name,
    due_date,
    condition_on_out="good",
    notes=None
):
    friend_query = """
        SELECT friend_id
        FROM friends
        WHERE full_name = :friend_name
    """
    friend_df = pd.read_sql(
        text(friend_query),
        con=engine,
        params={"friend_name": friend_name}
    )

    if friend_df.empty:
        print(f"Error: '{friend_name}' is not in your friends list.")
        return

    book_query = "SELECT title FROM books WHERE isbn = :isbn"
    book_df = pd.read_sql(
        text(book_query),
        con=engine,
        params={"isbn": str(isbn)}
    )

    if book_df.empty:
        print(f"Error: no book found with ISBN {isbn}.")
        return

    active_loan_query = """
        SELECT loan_id
        FROM loans
        WHERE isbn = :isbn AND status = 'out'
    """
    active_loan_df = pd.read_sql(
        text(active_loan_query),
        con=engine,
        params={"isbn": str(isbn)}
    )

    if not active_loan_df.empty:
        print(f"Loan denied: ISBN {isbn} is already checked out.")
        return

    insert_query = """
        INSERT INTO loans (
            isbn,
            friend_id,
            loan_date,
            due_date,
            condition_out,
            status,
            notes
        )
        VALUES (
            :isbn,
            :friend_id,
            CURRENT_DATE,
            :due_date,
            :condition_out,
            'out',
            :notes
        )
    """

    with engine.begin() as connection:
        connection.execute(text(insert_query), {
            "isbn": str(isbn),
            "friend_id": int(friend_df["friend_id"].iloc[0]),
            "due_date": due_date,
            "condition_out": condition_on_out,
            "notes": notes
        })

    print(f"Loan successfully recorded for {friend_name}.")

In [146]:
# to check 
create_loan_by_name(
    "9780441013593",
    "Semira",
    "2026-10-01"
)

Loan denied: ISBN 9780441013593 is already checked out.


### Update Functions

In [115]:
def update_friend(friend_id, full_name, email=None, phone=None, notes=None):
    """Update a friend's details (like fixing a name typo, email, or phone) by their friend_id."""
    
    update_query = """
        UPDATE friends 
        SET 
            full_name = COALESCE(:full_name, full_name),
            email = COALESCE(:email, email), 
            phone = COALESCE(:phone, phone), 
            notes = COALESCE(:notes, notes)
        WHERE friend_id = :friend_id
    """
    
    with engine.begin() as connection:
        connection.execute(text(update_query), {
            "friend_id": friend_id,
            "full_name": full_name,
            "email": email,
            "phone": phone,
            "notes": notes
        })
        
    print(f"Successfully updated information for Friend ID {friend_id}!")

In [116]:
# check
update_friend(2, full_name="Maya Chen-Smith")

Successfully updated information for Friend ID 2!


In [ ]:
def update_active_loan(isbn, condition_on_out=None, notes=None):
    update_query = """
        UPDATE loans
        SET
            condition_out = COALESCE(:condition_on_out, condition_out),
            notes = COALESCE(:notes, notes)
        WHERE isbn = :isbn AND status = 'out'
    """

    with engine.begin() as connection:
        result = connection.execute(text(update_query), {
            "isbn": isbn,
            "condition_on_out": condition_on_out,
            "notes": notes
        })

    if result.rowcount == 1:
        print(f"Successfully updated active loan for ISBN {isbn}.")
    else:
        print(f"No active loan found for ISBN {isbn}; nothing was updated.")

In [152]:
def update_returned_loan(loan_id, condition_on_return=None, notes=None):
    """Update a returned loan record using its unique loan_id."""

    update_query = """
        UPDATE loans
        SET
            condition_in = COALESCE(:condition_on_return, condition_in),
            notes = COALESCE(:notes, notes)
        WHERE loan_id = :loan_id AND status = 'returned'
    """

    with engine.begin() as connection:
        connection.execute(text(update_query), {
            "loan_id": loan_id,
            "condition_on_return": condition_on_return,
            "notes": notes
        })

    print(f"Successfully updated past loan record (Loan ID: {loan_id})!")

In [153]:
# check test the with returned loan ID 2.
update_returned_loan(
    2,
    condition_on_return="good",
    notes="Test from a cell"
)

Successfully updated past loan record (Loan ID: 2)!


In [ ]:
# display books from loans that have been returned.
returned_books_query = """
    SELECT
        l.loan_id,
        b.isbn,
        b.title,
        f.full_name AS borrower,
        l.loan_date,
        l.due_date,
        l.return_date,
        l.condition_in,
        l.notes
    FROM loans AS l
    JOIN books AS b ON b.isbn = l.isbn
    JOIN friends AS f ON f.friend_id = l.friend_id
    WHERE l.status = 'returned'
    ORDER BY l.return_date DESC
"""

returned_books = pd.read_sql(text(returned_books_query), con=engine)
display(returned_books)

,loan_id,isbn,title,borrower,loan_date,due_date,return_date,condition_in,notes
0,2,9780451524935,1984,Diego Alvarez,2026-05-01,2026-05-15,2026-05-14,good,Test from a cell
1,3,9781400033416,Beloved,Priya Nair,2026-04-10,2026-04-24,2026-04-20,fair,NaN
2,5,9780062316097,Sapiens,Yusuf Demir,2026-03-01,2026-03-15,2026-03-10,good,NaN


### Delete Operation

In [155]:
# Suggested replacements for the current sample_library schema.
def delete_friend(friend_name):
    friend_query = """
        SELECT friend_id
        FROM friends
        WHERE full_name = :friend_name
    """
    friend_df = pd.read_sql(
        text(friend_query),
        con=engine,
        params={"friend_name": friend_name}
    )

    if friend_df.empty:
        print(f"Error: '{friend_name}' was not found.")
        return

    friend_id = int(friend_df["friend_id"].iloc[0])

    loan_count_query = """
        SELECT COUNT(*) AS loan_count
        FROM loans
        WHERE friend_id = :friend_id
    """
    loan_count_df = pd.read_sql(
        text(loan_count_query),
        con=engine,
        params={"friend_id": friend_id}
    )

    if loan_count_df["loan_count"].iloc[0] > 0:
        print(
            f"Deletion denied: {friend_name} has loan history. "
            "Keep the friend record for history."
        )
        return

    with engine.begin() as connection:
        result = connection.execute(
            text("DELETE FROM friends WHERE friend_id = :friend_id"),
            {"friend_id": friend_id}
        )

    if result.rowcount == 1:
        print(f"Successfully removed {friend_name} from your friends list.")
    else:
        print(f"No friend record was removed for {friend_name}.")




In [156]:
# Test delete_friend with a temporary friend who has no loans.
test_friend_name = "Bambang Suharto"

create_friend(test_friend_name)
delete_friend(test_friend_name)

remaining_test_friend = pd.read_sql(
    text("SELECT * FROM friends WHERE full_name = :name"),
    con=engine,
    params={"name": test_friend_name}
)

if remaining_test_friend.empty:
    print("Test passed: this human was deleted.")
else:
    print("Test failed: the temporary friend still exists.")
display(remaining_test_friend)

Human name : Bambang Suharto successfully added!
Successfully removed Bambang Suharto from your friends list.
Test passed: this human was deleted.


,friend_id,full_name,email,phone,notes


In [161]:
def delete_loan(loan_id):
    loan_query = """
        SELECT status
        FROM loans
        WHERE loan_id = :loan_id
    """
    loan_df = pd.read_sql(
        text(loan_query),
        con=engine,
        params={"loan_id": loan_id}
    )

    if loan_df.empty:
        print(f"Error: Loan ID {loan_id} was not found.")
        return

    current_status = loan_df["status"].iloc[0]

    if current_status == "out":
        print(
            f"Deletion denied: Loan ID {loan_id} is still out. "
            "Return the book first."
        )
        return

    delete_query = "DELETE FROM loans WHERE loan_id = :loan_id"

    with engine.begin() as connection:
        result = connection.execute(
            text(delete_query),
            {"loan_id": loan_id}
        )

    if result.rowcount == 1:
        print(f"Successfully deleted loan record {loan_id}.")
    else:
        print(f"No loan record was deleted for ID {loan_id}.")

In [162]:
# Test delete_loan without deleting any real loan.
# Loan ID 1 is currently out, so deletion should be blocked.
delete_loan(1)

loan_one_after_test = pd.read_sql(
    text("SELECT loan_id, status FROM loans WHERE loan_id = :loan_id"),
    con=engine,
    params={"loan_id": 1}
)

if not loan_one_after_test.empty and loan_one_after_test.loc[0, "status"] == "out":
    print("Test passed: the active loan was not deleted.")
else:
    print("Test failed: the active loan changed unexpectedly.")

# Test the not-found branch.
delete_loan(999999)

Deletion denied: Loan ID 1 is still out. Return the book first.
Test passed: the active loan was not deleted.
Error: Loan ID 999999 was not found.
